# Structural Analysis of the OGBL-collab Dataset

This notebook performs a structural analysis of the OGBL-collab dataset.

In [1]:
import sys
import os
import time
import random
import json

import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from ogb.linkproppred import PygLinkPropPredDataset, Evaluator
from torch_geometric.utils import to_networkx, degree
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.loader import DataLoader

from torch.utils.data import random_split

import networkx as nx

c:\Master\GAI\gnn-over-squashing-cora\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sns.set_style("white")

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
    "axes.grid": False
})

os.makedirs("../../results/plots/ogbl-collab", exist_ok=True)
os.makedirs("../../results/tables/ogbl-collab", exist_ok=True)

## Dataset

In [3]:
_original_load = torch.load

def patched_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_load(*args, **kwargs)

torch.load = patched_load

dataset = PygLinkPropPredDataset(root="../../data", name="ogbl-collab")

data = dataset[0]

print(dataset)
print(data)

PygLinkPropPredDataset()
Data(num_nodes=235868, edge_index=[2, 2358104], x=[235868, 128], edge_weight=[2358104, 1], edge_year=[2358104, 1])


## Models

In [4]:
sys.path.append(os.path.abspath("../.."))

from src.models.link_prediction import GCN, GraphSAGE, GAT

from src.training.train import train_link_prediction
from src.training.evaluate import evaluate_link_prediction

from src.utils.seed import set_seed

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on {device}")

models = {
    "gcn": GCN,
    "graphsage": GraphSAGE,
    "gat": GAT
}

in_dim = dataset.num_features

hidden_dim = 64
epochs = 200
lr = 0.001
weight_decay = 1e-4
dropout = 0.5

layer_values = [2, 4, 6, 8]
seeds = [0, 7, 37, 42]

results = {m: {"layers": [], "acc_mean": [], "acc_std": [], "time": []} for m in models.keys()}

data = data.to(device)
split_edge = dataset.get_edge_split()

for model_name, Model in models.items():

    print(f"\nRunning experiments for {model_name.upper()}")

    for num_layers in layer_values:

        auc_runs = []
        hits_runs = []
        time_runs = []

        for seed in seeds:

            set_seed(seed)

            model = Model(in_dim, hidden_dim, num_layers, dropout).to(device)

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=lr,
                weight_decay=weight_decay
            )

            start = time.time()

            history = train_link_prediction(
                model,
                data,
                split_edge,
                optimizer,
                epochs=epochs,
            )

            elapsed = time.time() - start

            auc, hits = evaluate_link_prediction(
                model,
                data,
                split_edge,
            )

            auc_runs.append(auc)
            hits_runs.append(hits)
            time_runs.append(elapsed)

        # estadísticas
        auc_mean = np.mean(auc_runs)
        auc_std = np.std(auc_runs)

        hits_mean = np.mean(hits_runs)
        hits_std = np.std(hits_runs)

        results[model_name]["layers"].append(num_layers)
        results[model_name]["auc_mean"].append(auc_mean)
        results[model_name]["auc_std"].append(auc_std)
        results[model_name]["hits_mean"].append(hits_mean)
        results[model_name]["hits_std"].append(hits_std)
        results[model_name]["time"].append(np.mean(time_runs))

        print(
            f"Layers: {num_layers} | "
            f"AUC: {auc_mean:.4f} ± {auc_std:.4f}"
            f"hits@50: {hits_mean:.4f} ± {hits_std:.4f}"
        )